# EX: Bayesian Networks and Joint Factorization

In multi-domain command and control (C2) operations, commanders cannot rely on intuition to evaluate how environmental friction cascades into mission failure. A Bayesian Network models complex, multi-variable tactical engagements by decomposing full joint probability spaces into intuitive directed acyclic graphs (DAGs) and localized conditional probability tables (CPTs).

In this exercise, you will programmatically model the theater air-strike planning network from CS471 Quiz 5:

$$W \to V, \quad W \to C, \quad V \to M, \quad C \to M$$

Where:

$W$: Target Sector Weather ($\text{Clear}$ vs. $\text{Severe}$)

$V$: Visual Sensor Visibility ($\text{Good}$ vs. $\text{Poor}$)

$C$: SATCOM Communications Link ($\text{Reliable}$ vs. $\text{Disrupted}$)

$M$: Strike Mission Outcome ($\text{Success}$ vs. $\text{Abort}$)

You will implement the Conditional Probability Tables (CPTs), factor the joint distribution using the Bayesian network chain rule, verify conditional independence between visibility and communications given weather, and generate the full 16-row joint probability table.

Lab steps:

* Define Conditional Probability Tables (CPTs) - Represent the localized conditional distributions for $P(W)$, $P(V \mid W)$, $P(C \mid W)$, and $P(M \mid V, C)$ as nested Python dictionaries.

* Implement Joint Probability Factorization - Construct a function compute_joint_probability(w, v, c, m) that applies the Bayesian network chain rule: $$P(W, V, C, M) = P(W) \cdot P(V \mid W) \cdot P(C \mid W) \cdot P(M \mid V, C)$$

* Verify Conditional Independence - Demonstrate that once Weather ($W$) is known, Visibility ($V$) and SATCOM ($C$) factor independently:

$$P(V, C \mid W) = P(V \mid W) \cdot P(C \mid W)$$

* Exhaustive Joint Distribution Generation - Iterate through all 16 atomic configurations of the 4 binary variables, populate a Pandas DataFrame, and verify that the probabilities sum to exactly 1.0000.



In [1]:
import itertools
import pandas as pd

# -------------------------------------------------------------------------
# Step 1: Define Conditional Probability Tables (CPTs)
# -------------------------------------------------------------------------
# P(Weather): Prior over target sector weather
p_W = {
    'Clear': 0.80, 
    'Severe': 0.20
}

# P(Visibility | Weather): Optical sensor performance given weather
p_V_given_W = {
    'Clear':  {'Good': 0.90, 'Poor': 0.10},
    'Severe': {'Good': 0.30, 'Poor': 0.70}
}

# P(SATCOM | Weather): Satellite communications link given weather
p_C_given_W = {
    'Clear':  {'Reliable': 0.85, 'Disrupted': 0.15},
    'Severe': {'Reliable': 0.40, 'Disrupted': 0.60}
}

# P(Mission | Visibility, SATCOM): Strike success given sensor and comm states
p_M_given_VC = {
    ('Good', 'Reliable'):   {'Success': 0.95, 'Abort': 0.05},
    ('Good', 'Disrupted'):  {'Success': 0.60, 'Abort': 0.40},
    ('Poor', 'Reliable'):   {'Success': 0.50, 'Abort': 0.50},
    ('Poor', 'Disrupted'):  {'Success': 0.10, 'Abort': 0.90}
}

# -------------------------------------------------------------------------
# Step 2: Implement Joint Factorization Chain Rule
# -------------------------------------------------------------------------
def compute_joint_probability(w, v, c, m):
    """
    Computes P(W, V, C, M) = P(W) * P(V|W) * P(C|W) * P(M|V,C)
    """
    prob_w = p_W[w]
    prob_v = p_V_given_W[w][v]
    prob_c = p_C_given_W[w][c]
    prob_m = p_M_given_VC[(v, c)][m]
    return prob_w * prob_v * prob_c * prob_m

# Compute scenario: P(Clear, Good, Reliable, Success)
target_p = compute_joint_probability('Clear', 'Good', 'Reliable', 'Success')
print("=" * 65)
print(f"Joint Probability P(Clear, Good, Reliable, Success): {target_p:.4f} ({target_p * 100:.2f}%)")
print("=" * 65 + "\n")

# -------------------------------------------------------------------------
# Step 3: Verify Conditional Independence: V _|_ C | W
# -------------------------------------------------------------------------
# Under W = Clear, P(V=Good, C=Reliable | W=Clear) = P(V=Good | W=Clear) * P(C=Reliable | W=Clear)
p_v_good_clear = p_V_given_W['Clear']['Good']
p_c_rel_clear = p_C_given_W['Clear']['Reliable']
product_cond = p_v_good_clear * p_c_rel_clear

print("=== VERIFYING CONDITIONAL INDEPENDENCE: V _|_ C | W ===")
print(f"P(V=Good | W=Clear):                     {p_v_good_clear:.2f}")
print(f"P(C=Reliable | W=Clear):                 {p_c_rel_clear:.2f}")
print(f"P(V=Good, C=Reliable | W=Clear) Product: {product_cond:.4f} ({product_cond * 100:.2f}%)")
print("Status: Conditional independence holds by graphical d-separation.\n")

# -------------------------------------------------------------------------
# Step 4: Generate Full Joint Distribution Table
# -------------------------------------------------------------------------
rows = []
variable_domains = [
    ['Clear', 'Severe'],         # Weather
    ['Good', 'Poor'],             # Visibility
    ['Reliable', 'Disrupted'],    # SATCOM
    ['Success', 'Abort']          # Mission
]

for w, v, c, m in itertools.product(*variable_domains):
    p_val = compute_joint_probability(w, v, c, m)
    rows.append({
        'Weather': w, 
        'Visibility': v, 
        'SATCOM': c, 
        'Mission': m, 
        'Joint_Probability': p_val
    })

df_joint = pd.DataFrame(rows)

print("=== FULL FACTORED JOINT PROBABILITY TABLE (First 8 Configurations) ===")
print(df_joint.head(8).to_string(index=False))

total_prob_mass = df_joint['Joint_Probability'].sum()
print("\n" + "-" * 65)
print(f"Sum of all 16 atomic joint probabilities: {total_prob_mass:.4f}")
print("-" * 65)



Joint Probability P(Clear, Good, Reliable, Success): 0.5814 (58.14%)

=== VERIFYING CONDITIONAL INDEPENDENCE: V _|_ C | W ===
P(V=Good | W=Clear):                     0.90
P(C=Reliable | W=Clear):                 0.85
P(V=Good, C=Reliable | W=Clear) Product: 0.7650 (76.50%)
Status: Conditional independence holds by graphical d-separation.

=== FULL FACTORED JOINT PROBABILITY TABLE (First 8 Configurations) ===
Weather Visibility    SATCOM Mission  Joint_Probability
  Clear       Good  Reliable Success             0.5814
  Clear       Good  Reliable   Abort             0.0306
  Clear       Good Disrupted Success             0.0648
  Clear       Good Disrupted   Abort             0.0432
  Clear       Poor  Reliable Success             0.0340
  Clear       Poor  Reliable   Abort             0.0340
  Clear       Poor Disrupted Success             0.0012
  Clear       Poor Disrupted   Abort             0.0108

-----------------------------------------------------------------
Sum of all 16 at


## Interpreting the Results

Combinatorial Parameter Reduction: An unconstrained joint distribution table over 4 binary variables requires $2^4 - 1 = 15$ independent parameters stored in memory. By exploiting the conditional independence properties encoded in the DAG, our model required only 9 local parameters ($1 + 2 + 2 + 4 = 9$), achieving significant computational and memory savings while preventing overfitting.

Dominant Mission Profile: Under optimal sector conditions ($W = \text{Clear}, V = \text{Good}, C = \text{Reliable}$), the joint probability of successful strike execution is $0.5814$ ($58.14\%$). This value represents the combined probability of favorable weather ($0.80$), effective optical tracking ($0.90$), dependable communications ($0.85$), and terminal strike execution ($0.95$):


$$0.80 \times 0.90 \times 0.85 \times 0.95 = 0.5814$$

Mathematical Completeness: The sum of all 16 atomic configurations across the joint table sums to exactly $1.0000$. This confirms that the Bayesian network chain rule factors the distribution without probability leakage or distortion.